# SpikingJelly

SpikingJelly is an open-source deep learning framework for Spiking Neural Networks (SNNs) based on PyTorch.

NIR exchange support is available in `spikingjelly.activation_based.nir_exchange` (GitHub master / version ≥ 0.0.0.0.15).

**Install (if not already installed):**
```bash
pip install spikingjelly nirtorch nir
# Or for the latest version with nir_exchange:
# pip install git+https://github.com/fangwei123456/spikingjelly.git
```

## Export a NIR graph from SpikingJelly

Use `export_to_nir` to convert a SpikingJelly model to a `nir.NIRGraph`.
An example input tensor is required so that the exporter can infer the shape of each layer.

In [ ]:
import torch
import torch.nn as nn

import nir
from spikingjelly.activation_based import layer, neuron, functional
from spikingjelly.activation_based.nir_exchange import export_to_nir

# Build a simple feed-forward SNN: Linear → LIF → Linear → LIF
net = nn.Sequential(
    layer.Linear(8, 16),
    neuron.LIFNode(tau=2.0, v_threshold=1.0, v_reset=0.0),
    layer.Linear(16, 4),
    neuron.LIFNode(tau=2.0, v_threshold=1.0, v_reset=0.0),
)

# Example input: (batch=1, features=8) for a single time step
example_input = torch.zeros(1, 8)

# Export to NIR (dt=1e-4 seconds is the default, consistent with other NIR frameworks)
nir_graph = export_to_nir(net, example_input, dt=1e-4)

print("NIR graph nodes:")
for name, node in nir_graph.nodes.items():
    print(f"  {name}: {type(node).__name__}")
print("NIR graph edges:", nir_graph.edges)

In [ ]:
# Save the NIR graph to an HDF5 file
nir.write("spikingjelly_model.nir", nir_graph)
print("Saved NIR graph to spikingjelly_model.nir")

## Import a NIR graph to SpikingJelly

Use `import_from_nir` to reconstruct a SpikingJelly model from a `nir.NIRGraph`.
The result is a `torch.fx.GraphModule` with the same structure and parameters as the original.

In [ ]:
from spikingjelly.activation_based.nir_exchange import import_from_nir

# Load the NIR graph from file
loaded_graph = nir.read("spikingjelly_model.nir")

# Import back to SpikingJelly
imported_net = import_from_nir(loaded_graph, dt=1e-4, step_mode="s")
print("Imported model:")
print(imported_net)

### Verify round-trip numerical parity

We run the same random input through both models and compare outputs.
Note: both models must be reset before each forward pass to clear neuron state.

In [ ]:
import torch

net.eval()
imported_net.eval()

x = torch.randn(1, 8)

functional.reset_net(net)
with torch.no_grad():
    out_original = net(x)

functional.reset_net(imported_net)
with torch.no_grad():
    out_imported = imported_net(x)

print("Original output: ", out_original)
print("Imported output: ", out_imported)
print("Outputs match:   ", torch.allclose(out_original, out_imported, atol=1e-5))

## Constructing a NIR graph manually and importing to SpikingJelly

You can also build a `nir.NIRGraph` directly using NIR primitives and import it into SpikingJelly.
This is useful when you receive a NIR model from another framework.

In [ ]:
import numpy as np
import nir

dt = 1e-4  # time step in seconds
n_in, n_out = 4, 2

# Define NIR nodes
# nir.LIF uses continuous-time parameters:
#   tau = tau_sj * dt   (e.g. tau_sj=2.0  →  tau_nir = 2e-4)
#   r   = 1.0           (when decay_input=True in SpikingJelly)
manual_graph = nir.NIRGraph.from_list(
    nir.Affine(
        weight=np.random.randn(n_out, n_in).astype(np.float32),
        bias=np.zeros(n_out, dtype=np.float32),
    ),
    nir.LIF(
        tau=np.full(n_out, 2.0 * dt, dtype=np.float32),
        r=np.ones(n_out, dtype=np.float32),
        v_leak=np.zeros(n_out, dtype=np.float32),
        v_threshold=np.ones(n_out, dtype=np.float32),
        v_reset=np.zeros(n_out, dtype=np.float32),
    ),
)

print("Manual NIR graph nodes:")
for name, node in manual_graph.nodes.items():
    print(f"  {name}: {type(node).__name__}")

In [ ]:
from spikingjelly.activation_based.nir_exchange import import_from_nir
from spikingjelly.activation_based import functional

sj_model = import_from_nir(manual_graph, dt=1e-4, step_mode="s")
print("Imported SpikingJelly model:")
print(sj_model)

# Run a forward pass
x = torch.randn(1, n_in)
functional.reset_net(sj_model)
with torch.no_grad():
    out = sj_model(x)
print("Output shape:", out.shape)
print("Output:", out)

## Multi-step mode

SpikingJelly supports two step modes:
- `step_mode='s'` (default): each forward call processes one time step with input shape `(B, N)`.
- `step_mode='m'`: each forward call processes *T* time steps at once with input shape `(T, B, N)`.

NIR export works the same way regardless of step mode (the FX trace is step-mode-agnostic).
When importing from NIR, pass `step_mode='m'` to `import_from_nir` to get a multi-step model.

In [ ]:
import torch
import torch.nn as nn

import nir
from spikingjelly.activation_based import layer, neuron, functional
from spikingjelly.activation_based.nir_exchange import export_to_nir, import_from_nir

# Build a single-step model
net_s = nn.Sequential(
    layer.Linear(8, 8),
    neuron.LIFNode(tau=2.0, v_threshold=1.0, v_reset=0.0),
)

# Export to NIR (example_input is always single-step shape)
example_input = torch.zeros(1, 8)
nir_graph = export_to_nir(net_s, example_input, dt=1e-4)

# Import as a multi-step model
net_m = import_from_nir(nir_graph, dt=1e-4, step_mode='m')
print("Multi-step model:", net_m)

In [ ]:
# Verify that T sequential single-step passes equal one multi-step pass
torch.manual_seed(0)
T, B, N = 4, 2, 8
xs = [torch.randn(B, N) for _ in range(T)]

# Single-step: run T times
net_s.eval()
functional.reset_net(net_s)
outs_single = []
with torch.no_grad():
    for x in xs:
        outs_single.append(net_s(x))
out_single = torch.stack(outs_single, dim=0)  # (T, B, N)

# Multi-step: run once
x_multi = torch.stack(xs, dim=0)  # (T, B, N)
net_m.eval()
functional.reset_net(net_m)
with torch.no_grad():
    out_multi = net_m(x_multi)

print("Single-step output shape:", out_single.shape)
print("Multi-step  output shape:", out_multi.shape)
print("Outputs match:", torch.allclose(out_single, out_multi, atol=1e-5))

## ParametricLIFNode export

`neuron.ParametricLIFNode` has a learnable time constant τ. When exported to NIR,
τ is **frozen** at its current value and stored as a `nir.LIF` node.
The exported model can be imported back, but the τ will no longer be trainable.

In [ ]:
from spikingjelly.activation_based import layer, neuron
from spikingjelly.activation_based.nir_exchange import export_to_nir

dt = 1e-4
init_tau = 2.0

net_plif = nn.Sequential(
    layer.Linear(4, 4, bias=False),
    neuron.ParametricLIFNode(init_tau=init_tau, v_threshold=1.0, v_reset=0.0),
)

nir_graph = export_to_nir(net_plif, torch.zeros(1, 4), dt=dt)

for name, node in nir_graph.nodes.items():
    if isinstance(node, nir.LIF):
        print(f"Node '{name}': tau = {node.tau}  (expected ~{init_tau * dt:.2e})")

## Supported primitives

The following SpikingJelly ↔ NIR mappings are supported by `nir_exchange`:

| SpikingJelly | NIR | Notes |
|---|---|---|
| `layer.Linear` (no bias) | `nir.Linear` | |
| `layer.Linear` (with bias) | `nir.Affine` | |
| `nn.Linear` | `nir.Linear` / `nir.Affine` | |
| `layer.Conv2d` | `nir.Conv2d` | bias forced to zeros if `None` |
| `layer.AvgPool2d` | `nir.AvgPool2d` | |
| `layer.Flatten` | `nir.Flatten` | dim indices adjusted for T/B dims |
| `neuron.IFNode` | `nir.IF` | soft-reset maps to hard-reset with `v_reset=0` |
| `neuron.LIFNode` | `nir.LIF` | `tau_nir = tau_sj * dt` |
| `neuron.ParametricLIFNode` | `nir.LIF` | τ frozen at export time |

**Known limitations:**
- `nir.CubaLIF`, `nir.I`, `nir.LI`, `nir.Delay` have no SpikingJelly equivalent and are not importable.
- Heterogeneous per-neuron parameters (different `v_threshold` or `v_reset` per neuron) are not supported on import.
- Surrogate gradient type is not preserved.
- Soft-reset (`v_reset=None`) is mapped to hard-reset with `v_reset=0.0` on export; this is a lossy conversion.